# Auroral Phenomena Observations and Geo-Located Social Media Records (2015–2016) Exploration with `mlcroissant`

This notebook provides a step-by-step workflow for loading and exploring the [FAIR² Auroral Phenomena Observations and Geo-Located Social Media Records (2015–2016)](https://sen.science/doi/10.71728/senscience.mf4b-abv7) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.mf4b-abv7/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# View dataset's summary metadata
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")


## 2. Data Overview

Explore available record sets (`@id`), their fields, and columns. For reproducibility, **all entities are referenced by their `@id`.**

_Let's list all record sets in the dataset and summarize their field structure._

In [ ]:
# List available record sets and their fields using the Croissant metadata
record_sets = [rs for rs in metadata.recordSets]
print(f"Total record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(no name)')}")
    fields = rs.get('fields', [])
    print(f"  Fields:")
    for field in fields:
        print(f"    - {field['@id']} ({field.get('name', '')}) [{field.get('dataType', '')}]")
    print()


## 3. Data Extraction

Load data from each record set into a pandas DataFrame for analysis. We use the record set and field `@id`s (as seen above) for referencing entities programmatically.

_Below is an automated approach to loading all record sets into DataFrames using their `@id` fields._

In [ ]:
# Extract all record sets into pandas DataFrames, using @id as keys
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

print("Loading data for record sets:")
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Loaded record set: {rs_id} (records: {len(df)})")
    except Exception as e:
        print(f"  Could not load record set {rs_id}: {e}")

# Print available DataFrames and columns per record set
print("\nColumns for each record set:")
for rs_id, df in dataframes.items():
    print(f"- {rs_id}: {df.columns.tolist()}")

# Show preview for the first record set (if available)
if len(dataframes) > 0:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nPreview of record set: {first_rs_id}")
    display(dataframes[first_rs_id].head())


## 4. Exploratory Data Analysis (EDA)

Apply basic EDA and processing on a selected record set. For demonstration, we:
- Select a numeric field, filter by a threshold,
- Normalize the numeric field,
- Optionally group by a categorical field.

You can adjust `record_set_id`, `numeric_field_id`, and `group_field_id` below to match the fields available in the selected record set.

In [ ]:
# ---- CONFIGURATION: adjust these to actual IDs from above output ----
# Sample choices based on typical aurora observation data (replace with real @ids as needed)
record_set_id = list(dataframes.keys())[0]  # Use first available as demo
numeric_field_id = None

# Try to find a plausible numeric field (e.g., 'score', 'votes', etc.)
for col in dataframes[record_set_id].columns:
    if 'score' in col or 'votes' in col or 'id' not in col.lower():
        # Checking dtype
        if pd.api.types.is_numeric_dtype(dataframes[record_set_id][col]):
            numeric_field_id = col
            break

if numeric_field_id is None:
    print("No numeric field found for demonstration. Please update 'numeric_field_id' manually.")
else:
    threshold = dataframes[record_set_id][numeric_field_id].dropna().quantile(0.75)  # 75% quantile as example
    filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field_id] > threshold].copy()
    print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f} (total: {len(filtered_df)})")
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Try to group by a categorical field (e.g., 'region', 'country', 'color', etc.)
    possible_group_fields = [col for col in dataframes[record_set_id].columns if col not in [numeric_field_id] and dataframes[record_set_id][col].dtype == 'object']
    group_field = possible_group_fields[0] if possible_group_fields else None

    if group_field:
        # Show means by group (for normalized field for demonstration)
        grouped = filtered_df.groupby(group_field)[f"{numeric_field_id}_normalized"].mean().reset_index()
        print(f"\nGrouped mean of normalized field by '{group_field}':")
        display(grouped.head())
    else:
        print("No suitable categorical field found for grouping.")


## 5. Visualization

Visualize distributions or relationships for the selected numeric and categorical fields. If you adjusted the fields in the previous section, update the plotting code accordingly.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in dataframes[record_set_id].columns:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[record_set_id][numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping was possible, show boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=dataframes[record_set_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=60)
        plt.show()
else:
    print("Cannot plot: Missing numeric field.")

## 6. Conclusion

This notebook demonstrated how to load, examine, and explore the publicly available FAIR² auroral observation dataset using the `mlcroissant` library. We programmatically referenced dataset entities by their `@id`, explored the field structure, loaded record sets into DataFrames, and performed basic EDA and visualization. For further analysis or modeling, you can use the code structure above to focus on specific record sets, numeric or categorical fields, and their `@id` programmatic access points.